# Automatic Differentiation with autograd

Pytorch uses Automatic Differentiation (AD) to compute the gradients.
<br>

**Autograd** is a reverse automatic differentiation system. Autograd records a graph recording of all operations that created the data as you execute operation, giving you a **Directed Acyclic Graph (DAG)** whose leaves are the input tensors and roots are the output tensors. By tracing the graph from roots to leaves, you can automatically compute the gradients using the chain rule.

## 1. Pytorch computation graphs

In [1]:
import torch

x = torch.tensor(2.0,requires_grad= True)
w = torch.tensor(3.0,requires_grad= True)
b = torch.tensor(1.0,requires_grad= True)

#Operations
y = w*x
z = y + b

print("Result:",z)


Result: tensor(7., grad_fn=<AddBackward0>)


The grad_fn = AddBackward means that the function which created that tensor was a addition fucntion

### Forward Pass and Backward Pass

- **Forward Pass**: During forward pass(executing the operation like y = w*x), Pytorch records the operations and the tensors involved, building the graph. Tensors from operation will be tracked and have a **grad_fn** that will reference the function that created the tensors,forming a backward chain links in the graph, Tensors created by users will usually have **grad_fn = None**

- **Backward Pass** : When you call **.backward()** on a scalar tensor(the final loss value). Autorgrad will travese this graph backwards from that node. It uses chain rule of calculus which is guide by **grad_fn** at each step , to compute gradients of the scalar output with respect to the tensors that were intially marked with **requires_grad = True(model parameters)**

# 2. Tensor and Gradient calculations(requires_grad)

##  2.1 The **requires_grad** attribute

Every Pytorch tensor posseses a boolean attribute called requires_grad. This attribute acts as a **flag** to signal to Autograd whether operations involving this tensor should be recorded for potential gradient computation later.

By default , when you create a tensor its **requires_grad** attribute is set to **false**

In [2]:
import torch

x = torch.tensor([[1,2,3],[1,2,3]])
print(f"X tensor : {x}")
print(f"X.requires_grad: {x.requires_grad}")


X tensor : tensor([[1, 2, 3],
        [1, 2, 3]])
X.requires_grad: False


In [3]:
y = torch.tensor([1,2,3],requires_grad= False)
print(f"y Tensor : {y}")
print(f"y.requires_grad: {y.requires_grad}")

y Tensor : tensor([1, 2, 3])
y.requires_grad: False


### 2.1.1 Enabiling Gradient tracking

In [4]:
# During tensor creation
z = torch.tensor([1.1,-2.5,3],requires_grad= True)
print(f"z Tensor : {z}")
print(f"z.requires_grad: {z.requires_grad}")

z Tensor : tensor([ 1.1000, -2.5000,  3.0000], requires_grad=True)
z.requires_grad: True


In [5]:
# After tensor creation
b = torch.tensor([1.2,2,3])
print(f"Tensor b before : {b}")
print(f"b.requires_grad(before) : {b.requires_grad}\n")

b.requires_grad_(True)

print(f"Tensor b after : {b}")
print(f"b.requires_grad(After)) : {b.requires_grad}")

Tensor b before : tensor([1.2000, 2.0000, 3.0000])
b.requires_grad(before) : False

Tensor b after : tensor([1.2000, 2.0000, 3.0000], requires_grad=True)
b.requires_grad(After)) : True


### Note:
Gradient computations is only meaningful when you have floating-point tensors, Attempting **requires_grad = True** on integer will result in a error

## 2.2 Propagation of **requires_grad**
If any input tensor participating in a operation has **requires_grad = True** then the output tensor resulting from that will also have **requires_grad = True**

In [6]:
x = torch.tensor(2.0) # input data, gradients not needed
w = torch.tensor(3.0,requires_grad= True)# weight parameter. tracking needed
b = torch.tensor(1.0,requires_grad= True)# Bias parameter, tracking needed

print(f"x requires_grad: {x.requires_grad}")
print(f"w requires_grad: {w.requires_grad}")
print(f"b requires_grad: {b.requires_grad}\n")
 
intermediate = x * w

print(f"Intermediate Result: {intermediate}")
print(f"intermediate.requires_grad: {intermediate.requires_grad}\n")

y = intermediate + b

print(f"y.requires_grad: {y.requires_grad}")


x requires_grad: False
w requires_grad: True
b requires_grad: True

Intermediate Result: 6.0
intermediate.requires_grad: True

y.requires_grad: True


Notice that even though **x** did not require gradients, because w required gradients, the result of **w * x (intermediate)** also requires gradients. Subsequently, since intermediate required gradients (and b also did), the final output y also has **requires_grad=True.**

## 2.3 The **grad_fn** Attribute

When a new tensor is created by an operation, and its **requires_grad is True**, PyTorch attaches a .grad_fn attribute to this new tensor. This attribute references the function **(like AddBackward0 or MulBackward0)** that performed the operation and knows how to compute the corresponding gradients during the backward pass.

Tensors created directly by the user **(like our x, w, and b examples above)** are "leaf" tensors in the graph. If they have **requires_grad=True,** their **.grad_fn** is None because they weren't created by a tracked operation within the graph. Tensors resulting from operations on tensors requiring gradients are "**non-leaf**" tensors and will have a **.grad_fn.**

In [7]:
print(f"x.grad_fn: {x.grad_fn}")
print(f"w.grad_fn: {w.grad_fn}")
print(f"b.grad_fn: {b.grad_fn}")
print(f"intermediate.grad_fn: {intermediate.grad_fn}") # Result of multiplication
print(f"y.grad_fn: {y.grad_fn}") # Result of addition

x.grad_fn: None
w.grad_fn: None
b.grad_fn: None
intermediate.grad_fn: <MulBackward0 object at 0x750b682a0eb0>
y.grad_fn: <AddBackward0 object at 0x750b682a0490>


# 3. Performing Backpropagation(**backward()**)

The **backward()** method when called on a t**ensor(final scalar loss value of our model)**, it initiates the computation of gradients throughout the computation graph using the chain rule. It calculates the  gradient of the tensor with respect to all the leaf node(tensors) in the graph that has **requires_grad = True**.

## 3.1 Initiating Gradient Calcualtion
You always call the **backward()** method on **scalar tensor**, which is usually the result of the loss function

In [8]:
x = torch.tensor(2.0, requires_grad=False)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# perform some operations
y = w*x + b
loss = y * y # scalar

#before backward pass

print(f"x.grad: {x.grad}")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}\n")

loss.backward()

# after backpropagation

print("After Backward pass\n")
print(f"x.grad: {x.grad}")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}")




x.grad: None
w.grad: None
b.grad: None



After Backward pass

x.grad: None
w.grad: 28.0
b.grad: 14.0


Here **backwards()** calculated the gradients : $ \frac{\partial{loss}}{\partial{w}} $ , $ \frac{\partial{loss}}{\partial{b}} $ and $ \frac{\partial{loss}}{\partial{x}} $

### 3.2 Why .backward() on Scalar?

Autograd is designed to compute the **Jacobian-vector product (JVP)**. When you call backward() on a scalar tensor 
L, it's implicitly equivalent to calling backward() with a starting gradient of 1 i.e $ \frac{\partial{loss}}{\partial{loss}} = 1 $ This allows PyTorch to compute the gradients for all parameters efficiently using the chain rule propagating backward from the scalar loss.

In [9]:
x_vector = torch.tensor([2.0, 4.0], requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y_non_scalar = w * x_vector + b # y_non_scalar is now a non-scalar tensor with two elements: [7.0, 13.0]

try:
    y_non_scalar.backward() # This will cause an error
except RuntimeError as e:
    print(f"Error calling backward() on non-scalar: {e}")

# To make it work, provide a gradient tensor matching y_non_scalar's shape
# This represents the gradient of some final loss w.r.t y.

grad_tensor = torch.tensor([1.1,1.7])

y_non_scalar.backward(gradient=grad_tensor)

print(f"Gradient for x_vector after y_non_scalar.backward(gradient=...): {x_vector.grad}")
print(f"Gradient for w after y_non_scalar.backward(gradient=...): {w.grad}")

Error calling backward() on non-scalar: grad can be implicitly created only for scalar outputs
Gradient for x_vector after y_non_scalar.backward(gradient=...): tensor([3.3000, 5.1000])
Gradient for w after y_non_scalar.backward(gradient=...): 9.0


It is important to know that *gradients* accumulates by default. If you call **backward()** multiple times without clearing the gradients, then the newly computed gradients will be added to the values already present in **.grad** attribute. So you need to explicitly zero the gradients before each backpropagation step.

# 4. Disabling Gradient Tracking 

There are some scenario where gradient tracking is unnecessary . Some of those scenario are:
1) **During inference** : When you are using model to predict on a new data , you are not upadting its weights so tracking it doesnt make sense.

2) **For Frezzing model parameters** : During fine-tuning, you want to freeze the initial layers of a pretrained model , and only train later layers for transfer learning.


## 4.1 Using **torch.no_grad()** Context Manager

Any code inside this block will have gradient tracking disabled, even if they originally had **require_grad = true**

In [14]:
x = torch.tensor([1.0],requires_grad= True)
w = torch.tensor([7.0],requires_grad= True)
b = torch.tensor([4.0],requires_grad= True)

print(f"requires_grad.x: {x.requires_grad}")
print(f"requires_grad.w: {w.requires_grad}")
print(f"requires_grad.b: {b.requires_grad}")

y = w*x + b 
print(f"requires_grad.y: {y.requires_grad}")
print(f"Gradient function: {y.grad_fn}")

with torch.no_grad():
    z = x*w + b
    print(f"requires_grad.z: {z.requires_grad}")
    print(f"grad_fn.z:{z.grad_fn}")

# outside the block

print(f"requires_grad.z: {z.requires_grad}")
print(f"grad_fn.z:{z.grad_fn}")

requires_grad.x: True
requires_grad.w: True
requires_grad.b: True
requires_grad.y: True
Gradient function: <AddBackward0 object at 0x750b682a0eb0>
requires_grad.z: False
grad_fn.z:None
requires_grad.z: False
grad_fn.z:None


## 4.1 Using the **.detach()** method

This method creates a new tensor that shares the same underlying data storage as the original tensor but is explicitly detached from the current computation graph. 

In [23]:
a = torch.tensor([1.0,2,3],requires_grad= True)
b = torch.tensor([4.0],requires_grad=True)

c = a*b

print(f"requires_grad for c:{c.requires_grad}")
print(f"grad_fn for c: {c.grad_fn}")

d = c.detach()

#After detaching
print(f"\nrequires_grad for d:{d.requires_grad}")
print(f"grad_fn for d: {d.grad_fn}")

#Now original c remains the same

print(f"\nrequires_grad for c:{c.requires_grad}")
print(f"grad_fn for c: {c.grad_fn}")

# Any operation with d will not be tracked

e = d * 2

print(f"\nrequires_grad for e:{e.requires_grad}")
print(f"grad_fn for e: {e.grad_fn}")

requires_grad for c:True
grad_fn for c: <MulBackward0 object at 0x750b682a0460>

requires_grad for d:False
grad_fn for d: None

requires_grad for c:True
grad_fn for c: <MulBackward0 object at 0x750a8d91a410>

requires_grad for e:False
grad_fn for e: None


# 5. Gradient Accumulation

When you call **.backward()** multiple times without clearing the gradients in between, PyTorch adds the newly computed gradients to the existing values stored in the .grad attribute of the leaf tensors (parameters).

In [26]:
import torch

# Create a tensor that requires gradients
x = torch.tensor([2.0], requires_grad=True)

# Perform some operations
y = x * x
z = y * 3 

z.backward(retain_graph=True) #for back to back backprop
print(f"After first backward pass, x.grad: {x.grad}")


z.backward() 

print(f"After second backward pass, x.grad: {x.grad}")

# Manually zero the gradient
x.grad.zero_()
print(f"After zeroing, x.grad: {x.grad}")


After first backward pass, x.grad: tensor([12.])
After second backward pass, x.grad: tensor([24.])
After zeroing, x.grad: tensor([0.])


You would generally use **optimizer.zero_grad()** inside the training loop, so that gradients from previous batch gets cleared out before computing for the new batch